# Storage — 30 days to fill, 65 to empty

A seasonal storage deal with **asymmetric rates**: it injects fast and withdraws slowly.
Change `INJ_DAYS` and `WDR_DAYS` in §1 and everything below follows.

The reason this needs its own notebook is that the rates are **whole clips per day**, so the
deal you describe is only priceable if the inventory grid can express it. 30/65 cannot be
expressed on the grid you would reach for by default, and getting it wrong is silent — the
model prices a different contract and reports nothing. §2 does the arithmetic and §6 measures
what the near-misses cost.

Conventions are in [docs/MODEL-CONVENTIONS.md](docs/MODEL-CONVENTIONS.md).

In [ ]:
import os, sys, time, warnings
from math import gcd

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import storage_model as sm

pd.set_option("display.width", 200, "display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (12, 3.4), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
warnings.filterwarnings("ignore", category=FutureWarning)

_missing = [n for n in ("Storage", "run_valuation", "resolve_grid",
                        "params_for_run_valuation") if not hasattr(sm, n)]
if _missing:
    raise RuntimeError(
        "This kernel is running an older copy of storage_model — missing "
        + ", ".join(_missing) + ". Restart the kernel (Kernel > Restart Kernel and Run All).")

SMOKE = os.environ.get("STORAGE_NOTEBOOK_SMOKE") == "1"
print(f"ready — storage_model from {sm.__file__}")

## 1. The deal

Physical terms first, in the units a term sheet would use. The clip size and state count are
derived from these in §2 rather than chosen — that is the whole point.

`DISCOUNT_RATE` stays at 0 here. Storage pays on injection and receives on withdrawal, so it
has no single funding direction and `borrow_rate`/`invest_rate` are refused for it by design;
a market discount curve is the supported route, and time value is `Products.ipynb`'s subject.

In [ ]:
INJ_DAYS      = 30                      # days to fill at maximum rate
WDR_DAYS      = 65                      # days to empty at maximum rate
WDR_MWH_DAY   = 10_000.0                # maximum withdrawal, MWh/day
CAPACITY      = WDR_MWH_DAY * WDR_DAYS  # working volume follows: 650,000 MWh

START, END    = "2027-01-01", "2027-12-31"
VAL_DATE      = pd.Timestamp("2026-06-01")
INJ_COST      = 0.0                     # EUR/MWh injected
WDR_COST      = 0.0                     # EUR/MWh withdrawn
START_FULL    = 0.0                     # fraction full at the start
END_FULL      = 0.0                     # and at the end

VOL           = 0.50
SMR           = 1.0
DISCOUNT_RATE = 0.0                     # see the note above
N_P           = 12 if SMOKE else 25     # price-tree half-width; checked in §6

INJ_MWH_DAY = CAPACITY / INJ_DAYS
print(f"working volume {CAPACITY:,.0f} MWh over {START} .. {END}, valued {VAL_DATE:%Y-%m-%d}")
print(f"  inject   up to {INJ_MWH_DAY:>9,.0f} MWh/day  -> {INJ_DAYS:3d} days to fill")
print(f"  withdraw up to {WDR_MWH_DAY:>9,.0f} MWh/day  -> {WDR_DAYS:3d} days to empty")
print(f"  ratio {INJ_MWH_DAY/WDR_MWH_DAY:.4f} — the grid has to be able to express it")

## 2. Sizing the grid

The model moves inventory in whole **clips**, at a whole number of clips per day:

```
days to fill  = n_states / inj_rate
days to empty = n_states / wdr_rate
```

so `inj_rate / wdr_rate` must equal `WDR_DAYS / INJ_DAYS` in integers. The smallest grid that
does it exactly is `lcm(INJ_DAYS, WDR_DAYS)` states — **390** for 30/65, against the 60 that
30/60 would need. Any smaller grid rounds, and `params_for_run_valuation` uses
`max(1, round(n_states / days))`, so the rounding is silent: you get a different contract
with no warning.

The cell below picks the exact grid and **raises** if the derived rates do not reproduce the
days asked for, so a mis-sized grid stops the notebook rather than pricing something else.

In [ ]:
def exact_states(inj_days, wdr_days, multiple=1):
    """Smallest state count (times `multiple`) that expresses both rates exactly."""
    return inj_days * wdr_days // gcd(inj_days, wdr_days) * multiple


def rates_for(n_states, inj_days=None, wdr_days=None):
    """The library's own conversion: max(1, round(n_states / days)), both sides."""
    inj_days = INJ_DAYS if inj_days is None else inj_days
    wdr_days = WDR_DAYS if wdr_days is None else wdr_days
    return (max(1, round(n_states / inj_days)), max(1, round(n_states / wdr_days)))


N_STATES = exact_states(INJ_DAYS, WDR_DAYS)
INJ_RATE, WDR_RATE = rates_for(N_STATES)
V_STEP = CAPACITY / N_STATES

_candidates = sorted({30, 60, 65, 90, 130, 195, 260, N_STATES, N_STATES * 2})
_rows = []
for _n in _candidates:
    _ir, _wr = rates_for(_n)
    _rows.append({"n_states": _n, "inj_rate": _ir, "wdr_rate": _wr,
                  "days to fill": _n / _ir, "days to empty": _n / _wr,
                  "fill error": _n / _ir - INJ_DAYS, "empty error": _n / _wr - WDR_DAYS})
_grid = pd.DataFrame(_rows)
_grid["exact"] = (_grid["fill error"].abs() < 1e-9) & (_grid["empty error"].abs() < 1e-9)
display(_grid.style.hide(axis="index").format({
    "days to fill": "{:.2f}", "days to empty": "{:.2f}",
    "fill error": "{:+.2f}", "empty error": "{:+.2f}"})
    .apply(lambda r: ["background-color: #e8f5e9" if r["exact"] else "" for _ in r], axis=1)
    .set_caption(f"Grid sizes for {INJ_DAYS}/{WDR_DAYS} — errors in days. Exact needs a "
                 f"multiple of lcm({INJ_DAYS},{WDR_DAYS}) = {exact_states(INJ_DAYS, WDR_DAYS)}"))

_fill, _empty = N_STATES / INJ_RATE, N_STATES / WDR_RATE
if abs(_fill - INJ_DAYS) > 1e-9 or abs(_empty - WDR_DAYS) > 1e-9:
    raise AssertionError(
        f"the grid cannot express {INJ_DAYS}/{WDR_DAYS}: {N_STATES} states give "
        f"{_fill:.2f}/{_empty:.2f} days. Raise N_STATES to a multiple of "
        f"{exact_states(INJ_DAYS, WDR_DAYS)}.")
print(f"chosen grid: {N_STATES} states of {V_STEP:,.2f} MWh")
print(f"  inj_rate {INJ_RATE:2d} clips/day = {INJ_RATE*V_STEP:9,.0f} MWh/day -> {_fill:.2f} days to fill")
print(f"  wdr_rate {WDR_RATE:2d} clips/day = {WDR_RATE*V_STEP:9,.0f} MWh/day -> {_empty:.2f} days to empty")

## 3. The curve

A seasonal shape, dear in winter and cheap in summer, so there is a spread for the store to
capture. Swap in a real curve — a `curve.csv` frame or a quote row, as `Products.ipynb` §1
does — once you want a market number rather than a demonstration.

In [ ]:
SEASONAL_LEVEL, SEASONAL_SWING = 25.0, 6.0

_span = pd.date_range("2026-01-01", "2029-06-30", freq="D")
_doy = _span.dayofyear.values
CURVE = pd.Series(
    SEASONAL_LEVEL + SEASONAL_SWING * np.cos(2 * np.pi * (_doy - 15) / 365.25), index=_span)

_win = CURVE.loc[START:END]
fig, ax = plt.subplots()
ax.plot(_win.index, _win.values, color="black", lw=1.4)
ax.axhline(_win.mean(), color="tab:red", ls="--", lw=1,
           label=f"window mean {_win.mean():.2f} EUR/MWh")
ax.fill_between(_win.index, _win.values, _win.mean(), where=_win.values < _win.mean(),
                color="tab:green", alpha=.18, label="below average — inject here")
ax.fill_between(_win.index, _win.values, _win.mean(), where=_win.values > _win.mean(),
                color="tab:orange", alpha=.18, label="above average — withdraw here")
ax.set_ylabel("EUR/MWh"); ax.legend(loc="best", fontsize=8)
ax.set_title(f"Forward curve over the deal window, {START} .. {END}")
plt.tight_layout(); plt.show()
print(f"window: min {_win.min():.2f}  mean {_win.mean():.2f}  max {_win.max():.2f} EUR/MWh  "
      f"(summer-winter spread {_win.max()-_win.min():.2f})")

## 4. Price it

`run_valuation` with the derived rates. `intrinsic` is the value of the schedule you could
lock in today against the forward curve; `extrinsic` is what re-optimising as prices move
adds on top.

In [ ]:
def price(n_states=None, n_p=None, run_intrinsic=True):
    """Value the deal on a given grid. Rates are always derived, never assumed."""
    n_states = N_STATES if n_states is None else n_states
    inj_rate, wdr_rate = rates_for(n_states)
    v_step = CAPACITY / n_states
    params = dict(
        product_type="storage", valDate=VAL_DATE, storageStart=START, storageEnd=END,
        capacity_mwh=CAPACITY, daily_max=inj_rate * v_step, clips_per_day=inj_rate,
        inj_rate=inj_rate, wdr_rate=wdr_rate,
        initial_inv_clips=int(round(START_FULL * n_states)),
        terminal_inv_clips=int(round(END_FULL * n_states)),
        inj_cost=INJ_COST, wdr_cost=WDR_COST, vol=VOL, sMR=SMR,
        n_p_full=N_P if n_p is None else n_p, run_intrinsic=run_intrinsic,
        discount_rate=DISCOUNT_RATE, daily_curve=CURVE)
    t0 = time.perf_counter()
    model, result = sm.run_valuation(None, params)
    secs = time.perf_counter() - t0

    n = model.n_t
    moved = model.prob[:n] * model.strat[:n] * model.v_step
    injected = np.clip(moved, 0, None).sum(axis=(1, 2))
    withdrawn = -np.clip(moved, None, 0).sum(axis=(1, 2))
    return model, result, dict(
        n_states=n_states, inj_rate=inj_rate, wdr_rate=wdr_rate, v_step=v_step,
        fill=n_states / inj_rate, empty=n_states / wdr_rate, secs=secs,
        v0=float(model.v[0, model.n_p, model.initial_state]),
        injected=injected, withdrawn=withdrawn,
        dates=pd.DatetimeIndex(model.date_span)[:n])


MODEL, RESULT, DIAG = price()
print(f"value {DIAG['v0']:>14,.0f} EUR      {DIAG['v0']/CAPACITY:7.4f} EUR per MWh of capacity")
print(f"  intrinsic {RESULT['intrinsic']:8.4f} EUR/MWh cycled")
print(f"  extrinsic {RESULT['extrinsic']:8.4f}")
print(f"  cycled {DIAG['injected'].sum():,.0f} MWh in, {DIAG['withdrawn'].sum():,.0f} MWh out "
      f"({DIAG['injected'].sum()/CAPACITY:.2f} turns)   {DIAG['secs']:.1f}s")

if DIAG["injected"].max() > INJ_RATE * V_STEP + 1e-6:
    raise AssertionError("injection breached its daily cap")
if DIAG["withdrawn"].max() > WDR_RATE * V_STEP + 1e-6:
    raise AssertionError("withdrawal breached its daily cap")
print(f"rate caps hold: peak in {DIAG['injected'].max():,.0f} of {INJ_RATE*V_STEP:,.0f}, "
      f"peak out {DIAG['withdrawn'].max():,.0f} of {WDR_RATE*V_STEP:,.0f} MWh/day")

## 5. The schedule

The expected inventory path and the daily rates behind it. Two things to look for: the
inventory never exceeds capacity, and neither rate exceeds its cap — the dashed lines.

Note that the store injects on **more** than `INJ_DAYS` days. The 30/65 figures are maximum
rates, not day counts: over a year the optimiser need not run flat out, and it can part-cycle
more than once.

In [ ]:
_inv = np.cumsum(DIAG["injected"] - DIAG["withdrawn"])
_active = (DIAG["dates"] >= pd.Timestamp(START)) & (DIAG["dates"] <= pd.Timestamp(END))

fig, ax = plt.subplots(2, 1, figsize=(12, 5.4), sharex=True)
ax[0].fill_between(DIAG["dates"], _inv, 0, color="tab:blue", alpha=.25)
ax[0].plot(DIAG["dates"], _inv, color="tab:blue", lw=1.3)
ax[0].axhline(CAPACITY, color="tab:red", ls="--", lw=1, label=f"capacity {CAPACITY:,.0f} MWh")
ax[0].set_ylabel("MWh in store"); ax[0].legend(fontsize=8)
ax[0].set_title(f"Expected inventory — {INJ_DAYS} days to fill, {WDR_DAYS} to empty, "
                f"{N_STATES} states")

ax[1].bar(DIAG["dates"], DIAG["injected"], width=1.0, color="tab:green", label="injected")
ax[1].bar(DIAG["dates"], -DIAG["withdrawn"], width=1.0, color="tab:orange", label="withdrawn")
ax[1].axhline(INJ_RATE * V_STEP, color="tab:green", ls="--", lw=.9)
ax[1].axhline(-WDR_RATE * V_STEP, color="tab:orange", ls="--", lw=.9)
ax[1].axhline(0, color="black", lw=.8)
ax[1].set_ylabel("MWh/day"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

_det, _, _dd = price(run_intrinsic=False, n_p=0)
_di, _dw = _dd["injected"], _dd["withdrawn"]
_dates = _dd["dates"]
print(f"deterministic schedule (the intrinsic strategy):")
print(f"  injects on   {int((_di > 1).sum()):3d} days, {_dates[_di > 1].min():%d %b} .. "
      f"{_dates[_di > 1].max():%d %b}, peak {_di.max():,.0f} MWh/day")
print(f"  withdraws on {int((_dw > 1).sum()):3d} days, {_dates[_dw > 1].min():%d %b} .. "
      f"{_dates[_dw > 1].max():%d %b}, peak {_dw.max():,.0f} MWh/day")
_dinv = np.cumsum(_di - _dw)
print(f"  peak inventory {_dinv.max():,.0f} of {CAPACITY:,.0f} MWh; ends at {_dinv[-1]:,.0f}")

## 6. What a mis-sized grid costs

The reason §2 raises rather than warns. Each row below prices the *same* physical deal on a
different grid; the rates it can express change, so the contract silently changes with it.
The value column is not a convergence study — a coarser grid here is not a rougher
approximation of the same deal, it is a **different deal**.

The last column is the honest one: how far the priced contract is from the one you described.

In [ ]:
_rows = []
for _n in ([N_STATES] if SMOKE else [60, 130, 195, 260, N_STATES]):
    _m, _r, _d = price(n_states=_n, n_p=N_P)
    _rows.append({
        "n_states": _n, "rates": f"{_d['inj_rate']}/{_d['wdr_rate']}",
        "fill days": _d["fill"], "empty days": _d["empty"],
        "MWh/day in": _d["inj_rate"] * _d["v_step"],
        "MWh/day out": _d["wdr_rate"] * _d["v_step"],
        "value EUR": _d["v0"], "EUR/MWh cap": _d["v0"] / CAPACITY,
        "vs exact": _d["v0"] / [r for r in _rows if r["n_states"] == N_STATES][0]["value EUR"] - 1
        if any(r["n_states"] == N_STATES for r in _rows) else np.nan,
        "secs": _d["secs"]})
_exact_value = [r for r in _rows if r["n_states"] == N_STATES][0]["value EUR"]
for _r in _rows:
    _r["vs exact"] = _r["value EUR"] / _exact_value - 1.0
_cost = pd.DataFrame(_rows)
display(_cost.style.hide(axis="index").format({
    "fill days": "{:.2f}", "empty days": "{:.2f}", "MWh/day in": "{:,.0f}",
    "MWh/day out": "{:,.0f}", "value EUR": "{:,.0f}", "EUR/MWh cap": "{:.4f}",
    "vs exact": "{:+.2%}", "secs": "{:.1f}"})
    .set_caption(f"The same {INJ_DAYS}/{WDR_DAYS} deal on different grids — only "
                 f"{N_STATES} states prices the contract asked for"))
print(f"the exact grid costs {_cost['secs'].max():.1f}s at n_p {N_P}, so there is no reason "
      f"to accept a near miss.")

## Traps

- **The grid, not the days, decides the contract.** `max(1, round(n_states / days))` rounds
  silently. 30/65 needs a multiple of 390 states; the 60 that suits 30/60 prices a 30/60
  deal instead, and nothing says so. §2 raises on the mismatch — keep that check.
- **`inj_days` means two things.** In `resolve_grid` it is the inventory **state count**; in
  `params_for_run_valuation` it is **days to fill**. This notebook never passes it, deriving
  `inj_rate`/`wdr_rate` explicitly instead.
- **Rates are maximums, not schedules.** The store will use more than 30 injection days if
  that is worth more, and may part-cycle several times over a year.
- **Storage has no single funding direction.** It pays on injection and receives on
  withdrawal, so `borrow_rate`/`invest_rate` are refused for it. Use a market `discount_rate`
  or build `d_curve` directly.
- **`profiled()` is undefined here.** Value per *net* exercised MWh divides by roughly zero
  for a cycling deal, which is why it raises. The per-MWh figures above divide by capacity,
  which is stated rather than implied.